# 03. Выводы и бизнес-рекомендации

В этом ноутбуке превращаю находки из 02 в три конкретные рекомендации, которые можно отдать продуктовой команде. Никакой математики, только бизнес-язык и числа.

Если совсем коротко: ключевое окно для борьбы за повторную покупку это первые 30 дней, ключевая аудитория для кампаний это клиенты с низким первым чеком, ключевая аномалия это декабрьские когорты, которые надо обрабатывать отдельно.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

customers = pd.read_parquet('../data/customers_labeled.parquet')
retention = pd.read_parquet('../data/cohort_retention.parquet')
print(f'Клиентов в сегментации: {len(customers):,}')

## Краткое резюме того, что нашли

| Что нашли | Метрика | Что это значит для бизнеса |
|---|---|---|
| Retention сильно проседает в первый месяц | со 100% до 22% | Окно для action: первые 30 дней |
| Размер первого чека связан с возвратом | t-тест Уэлча, p << 0.001 | Можно скорить клиентов сразу после первой покупки |
| Декабрьские когорты ведут себя хуже среднего | разница в M+1 видна на тепловой карте | Не лить retention-бюджет на 'подарочных' в январе |

## Рекомендация 1. Триггерные кампании реактивации только для низкого чека

Что предлагаю. Запускать персональную email- или push-кампанию с промокодом на вторую покупку, но только для клиентов с первым чеком ниже медианы. Окно отправки: первые 14 дней после первой покупки.

Почему именно так. У сегмента 'высокий чек' доля возврата и без вмешательства высокая, и тратить на них retention-бюджет это деньги в никуда. У сегмента 'низкий чек' возвратность ниже, и потенциальный прирост retention имеет максимальную ценность. Окно 'первые 14 дней' выбрано потому, что критическое падение случается в первый месяц, и достучаться до клиента надо до того, как он 'остыл' окончательно.

In [ ]:
low_check = customers[customers['check_segment'].str.startswith('Низкий')]
high_check = customers[customers['check_segment'].str.startswith('Высокий')]

lift_potential = high_check['returned'].mean() - low_check['returned'].mean()

print(f'Размер целевого сегмента (низкий чек):  {len(low_check):,} клиентов')
print(f'Текущая доля возврата в сегменте:        {low_check["returned"].mean()*100:.1f}%')
print(f'Доля возврата в сегменте "высокий чек":  {high_check["returned"].mean()*100:.1f}%')
print(f'Разрыв (потенциал апсайда):              {lift_potential*100:.1f} п.п.')

Как мерить успех кампании. Делим целевой сегмент случайно пополам, одна половина получает триггерное письмо, другая нет. Главная метрика: доля совершивших вторую покупку в окне 30 дней. Контрольная (guardrail) метрика: средний чек второй покупки и общая выручка с клиента за 60 дней. Это нужно, чтобы убедиться, что промо не съедает свою же экономику. Минимально детектируемый эффект: 2 п.п. абсолютных. Подробный расчёт размера выборки и валидация дизайна теста через симуляцию находятся в ноутбуке 05.

## Рекомендация 2. Декабрьских клиентов в отдельную воронку

Что предлагаю. Клиентов, у которых первая покупка пришлась на декабрь, исключать из стандартных январских и февральских кампаний реактивации. Вместо промокодов отправлять им мягкую 'приветственную' серию: контент о категориях, которых не было в первой покупке, без скидок.

Почему. Декабрьские когорты имеют специфический мотив покупки (подарок, не для себя). Стандартная кампания 'вернись с промо на свой любимый товар' им неактуальна, ведь они купили не для себя. Скидки на этот сегмент в январе это потерянные деньги в маркетинге.

Цифра, на которую опираюсь. На тепловой карте видно, что декабрьские когорты в M+1 (январь) возвращаются хуже сравнимых месяцев.

In [ ]:
december_mask = retention.index.month == 12
dec_retention_m1 = retention.loc[december_mask, 1].mean()
other_retention_m1 = retention.loc[~december_mask, 1].mean()

print(f'Retention M+1 у декабрьских когорт:  {dec_retention_m1:.1f}%')
print(f'Retention M+1 у остальных когорт:    {other_retention_m1:.1f}%')
print(f'Разница:                              {other_retention_m1 - dec_retention_m1:.1f} п.п.')

## Рекомендация 3. Первый чек как ранний скоринговый признак

Что предлагаю. Добавить в продуктовую аналитику 'первый чек выше или ниже медианы' как ранний предиктор LTV. Использовать в дашбордах онбординга и в моделях прогноза LTV (даже если просто как одну из фич, без сложного ML).

Почему. Уже на следующий день после первой покупки мы знаем, в какой группе клиент. Это даёт CRM-команде грубую сегментацию 'бесплатно', без накопления истории.

Чего точно не делать. Нельзя строить рекламную или продуктовую логику в духе 'низкий чек = плохой клиент, не показываем дорогие товары'. Это путь в самоисполняющееся пророчество. Сегментация полезна только для догоняющих активностей (реактивация, реклама), но не для дискриминации в самом продукте.

## Что есть в проекте дополнительно

Чтобы рекомендации стояли не только на статистике, к проекту приложены ещё три ноутбука с практическими расчётами:

- 04. LTV и unit-экономика. Считаю наблюдаемый LTV каждого клиента, сравниваю по сегментам, строю Парето по концентрации выручки и проверяю чувствительность экономики к стоимости привлечения.
- 05. Дизайн A/B-теста. Считаю необходимый размер выборки на каждую группу для MDE = 2 п.п., строю power curve и валидирую дизайн через Монте-Карло симуляцию (включая проверку доли ложных срабатываний).
- 06. RFM-сегментация. Делю всю базу клиентов по трём осям (Recency, Frequency, Monetary), назначаю понятные бизнес-имена ('Чемпионы', 'Под угрозой', 'Потерянные') и для каждого сегмента предлагаю конкретное действие.

## Что осталось за кадром

Честный список ограничений, чтобы не казалось, что проект закрывает всё на свете.

1. В датасете нет источника трафика и канала привлечения. На реальном маркетплейсе retention сильно зависит от того, пришёл клиент по органике, по рекламе или по реферальной программе. Здесь этого среза просто нет.
2. Большая доля выручки приходит от оптовых клиентов. У них поведение принципиально отличается от обычной розницы, и в идеале их надо вынести в отдельный анализ. Я этого не делал, чтобы не дробить аудиторию.
3. Не строил предиктивную модель возврата. Считаю, что для текущей задачи это лишнее: простая сегментация и t-тест отвечают на бизнес-вопрос, а интерпретируемая модель (например, логистическая регрессия) добавила бы немного, но потребовала бы вдвое больше времени на защиту.
4. t-тест и сегментация показывают связь, но не причинность. Доказать, что увеличение первого чека вызывает рост retention, а не наоборот, можно только A/B-тестом. Дизайн такого теста описан в ноутбуке 05.

## Финальная фраза

Самый дешёвый способ повысить retention это не пытаться удержать всех, а вычислить тех, кто вернётся и без вмешательства, и не тратить на них бюджет. Размер первого чека простой и ранний сигнал, который позволяет это сделать.